# Project Config

In [2]:
from pathlib import Path
import subprocess

In [54]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: mps


In [3]:
def make_dir(dir_path: Path):
    dir_path.mkdir(parents=True, exist_ok=True)

def file_structure(assets_dir: Path):
    make_dir(assets_dir)

    input_dir = assets_dir / "raw"
    make_dir(input_dir)
    queue_dir = assets_dir / "queue"
    make_dir(queue_dir)
    history_dir = assets_dir / "processed"
    make_dir(history_dir)
    processed_csv_dir = history_dir / "csv"
    make_dir(processed_csv_dir)

    return input_dir, queue_dir, history_dir
    
 
    # csv_dir = history_dir / "csv"
    # make_dir(csv_dir)

def clean_dir(dir_path: Path):
    for file in dir_path.glob("*"):
        if file.is_dir():
            clean_dir(file)
        else:
            file.unlink()
    dir_path.rmdir()


In [17]:
project_root = Path('__file__').parent
data_dir  = Path(project_root / "note_assets")

raw_dir, queue_dir, processed_dir = file_structure(data_dir)

In [6]:
clean_dir(data_dir)

# load data into input file

In [14]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download(
    "pad19tue/lecture-videos",
    output_dir=raw_dir,
    )
print("Path to dataset files:", path)

FileExistsError: output_dir is not empty: note_assets/raw. Set force_download=True to replace it.

In [ ]:
# move files to raw directory 

video_dir = raw_dir / "video"
print 
for mp4 in video_dir.glob("*.mp4"):
    mp4.rename(raw_dir / mp4.name)



# Video / Audio Processing 

In [16]:
import subprocess
import uuid

In [19]:
def chunk_media(source: str, output_dir: str, seconds: int = 30, video_id: str = None):
    source = Path(source)
    if video_id is None:
        data_id = uuid.uuid4()
    else:
        data_id = video_id
    output = Path(output_dir)
    pattern = output / f"{data_id}_%06d{source.suffix}"
    command = [
        "ffmpeg",
        "-y",
        "-i", str(source), #input file
        "-f", "segment", #format
        "-segment_time", str(seconds), #segment time
        "-reset_timestamps", "1", #reset timestamps
        "-c", "copy", #copy
        "-segment_list", str(output / f"{data_id}_segments.csv"),
        "-segment_list_type", "csv",
        str(pattern), #output pattern
    ]
    subprocess.run(command, check=True)
    return data_id

In [20]:
for file in raw_dir.glob("*.mp[34]"):
    video_id = file.stem
    chunk_media(file, queue_dir, video_id=video_id)
    file.rename(processed_dir / file.name)



ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.3.34.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.0.2_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      61.  1.102 / 61.  1.102
  libavcodec     63.  1.102 / 63.  1.102
  libavformat    63.  1.102 / 63.  1.102
  libavdevice    63.  1.102 / 63.  1.102
  libavfilter    12.  1.102 / 12.  1.102
  libswscale     10.  1.102 / 10.  1.102
  libswresample   7.  1.102 /  7.  1.102
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'note_assets/raw/theory_of_computation_high_res.mp4':
  Metadata:
    major_brand     : mp42
    minor_version   : 0
    compatible_brands: isomm

# Audio to Text Processing

This section explores whether splitting the audio into shunks and transcribing it makes it faster than processing the whole audio

In [36]:
import whisper
import time

In [55]:
model = whisper.load_model("small", device= "cuda" if torch.cuda.is_available() else "cpu")
audio_data = []

# transcribe chunked files

In [48]:
clips = [*queue_dir.glob("*.mp4")]
print(clips)
clip_data = {}
file_name = "cities_and_climate_high_res"
process_start = time.time()
print(f"Start transcribing chunks ...")
for i, clip in enumerate(clips):
    li = clip.stem.split("_")
    clip_id = li[-1]
    video_id= "_".join(li[:-1])
    if video_id != file_name:
        continue
    result = model.transcribe(str(clip), word_timestamps=True, fp16=False)
    clip_data= {
        "video_id": video_id,
        "clip_id": clip_id,
        "text": result["text"]
    }
    audio_data.append(clip_data)
print(f"Total time taken: {time.time() - process_start} seconds")



[PosixPath('note_assets/queue/phonetics_high_res_000009.mp4'), PosixPath('note_assets/queue/theory_of_computation_high_res_000017.mp4'), PosixPath('note_assets/queue/cryptocurrency_high_res_000040.mp4'), PosixPath('note_assets/queue/theory_of_computation_high_res_000003.mp4'), PosixPath('note_assets/queue/cryptocurrency_high_res_000054.mp4'), PosixPath('note_assets/queue/creating_breakthrough_products_MIT_000059.mp4'), PosixPath('note_assets/queue/creating_breakthrough_products_MIT_000071.mp4'), PosixPath('note_assets/queue/phonetics_high_res_000035.mp4'), PosixPath('note_assets/queue/phonetics_high_res_000021.mp4'), PosixPath('note_assets/queue/cryptocurrency_high_res_000068.mp4'), PosixPath('note_assets/queue/creating_breakthrough_products_MIT_000065.mp4'), PosixPath('note_assets/queue/team_dynamics_high_res_000050.mp4'), PosixPath('note_assets/queue/clinical_care_high_res_000148.mp4'), PosixPath('note_assets/queue/cryptocurrency_high_res_000083.mp4'), PosixPath('note_assets/queue/cr

In [52]:
audio_data.sort(key=lambda x: int(x["clip_id"]))
start = 0
end = 30
for item in audio_data:
    print(f"start: {start}, end: {end}, text: {item['text']}")
    start += 30
    end += 30
    

start: 0, end: 30, text:  And this is a shorter video lecture introducing my class on city to climate action. Or this is really kind of an exploration or maybe rationale perhaps. But why you should take this class, MIT 11.165 for undergraduates or 11.477 for graduate students. This video lecture really I'm going to use this.
start: 30, end: 60, text:  example of all the future video lectures that are to come. So for example, I'm not going to go over this slide. This slide has all the readings that are in the syllabus for today. I'll generally start my lectures with a slide like this, but I won't go into it because first, all these things are on the syllabus. Second, for all the students taking this class at MIT, you'll have the PDF slides available on the Canvas site. So you can go to the PDF slides and click on the links. You don't need to go clicking the links for the ones on the syllabus because you're
start: 60, end: 90, text:  have the syllabus, but if I bring in other reports or 

# Transcribe whole file

In [23]:
import time

# for demonstration purpose, we only use on sample video (cities_and_climate_high_res.mp4)
start = time.time()
print("Transcribing the whole video ...")
result = model.transcribe(str(processed_dir / "cities_and_climate_high_res.mp4"), word_timestamps=True, fp16=False)
print(f"Total time taken: {time.time() - start} seconds")

print(result)



Transcribing the whole video ...
Total time taken: 244.42412185668945 seconds
{'text': " And this is a shorter video lecture introducing my class on city to climate action. Or this is really kind of an exploration or maybe rationale perhaps. But why you should take this class, MIT 11.165 for undergraduates or 11.477 for graduate students. This video lecture really I'm going to use this as an example of all the future video lectures that are to come. So for example, I'm not going to go over this slide. This slide has all the readings that are in the syllabus for today. I'll generally start my lectures with a slide like this, but I won't go into it because first, all these things are on the syllabus. Second, for all the students taking this class at MIT, you'll have the PDF slides available on the campus site. So you can go to PDF slides and click on the links. You don't need to go click on the links for the one with the syllabus because you already have the syllabus. But for if I bring 

In [24]:
for segment in result["segments"]:
   print(f"Start: {segment['start']}, End: {segment['end']},{segment['text']}")


Start: 11.079999999999998, End: 15.76, And this is a shorter video lecture introducing my class on city to climate action.
Start: 16.44, End: 20.3, Or this is really kind of an exploration or maybe rationale perhaps.
Start: 20.88, End: 28.02, But why you should take this class, MIT 11.165 for undergraduates or 11.477 for graduate
Start: 28.02, End: 28.5, students.
Start: 29.22, End: 33.62, This video lecture really I'm going to use this as an example of all the future video
Start: 33.62, End: 34.78, lectures that are to come.
Start: 35.04, End: 37.54, So for example, I'm not going to go over this slide.
Start: 37.98, End: 40.48, This slide has all the readings that are in the syllabus for today.
Start: 40.9, End: 46.18, I'll generally start my lectures with a slide like this, but I won't go into it because first,
Start: 46.46, End: 47.92, all these things are on the syllabus.
Start: 48.38, End: 54.22, Second, for all the students taking this class at MIT, you'll have the PDF slides ava

# Embedder - Embedding Gemma 2 by google


In [56]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer


print(f"Using device: {device}")

model = SentenceTransformer(
    "google/embeddinggemma-2",
    device=device,
    model_kwargs={"torch_dtype": torch.float32},
    config_kwargs={"audio_config": None},
)

W1020 14:07:46.710000 48011 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Using device: mps


Loading weights: 100%|██████████| 624/624 [00:02<00:00, 307.83it/s]


In [ ]:
# video data preparation
import pandas as pd

video_metadata = []
for file in queue_dir.glob("*.csv"):
    l = file.stem.split("_")
    video_id = " ".join(l[:-1])
    df = pd.read_csv(file, header=None, names=["name", "start", "end"])
    for index, row in df.iterrows():
        video_metadata.append({
            "video_id": video_id,
            "clip_id": index,
            "modality": "video",
            "file_name": row["name"],
            "start": row["start"],
            "end": row["end"]
        })
file.rename(processed_dir / "csv" / file.name)
print(pd.DataFrame(video_metadata))



                            video_id  clip_id modality  \
0     theory of computation high res        0    video   
1     theory of computation high res        1    video   
2     theory of computation high res        2    video   
3     theory of computation high res        3    video   
4     theory of computation high res        4    video   
...                              ...      ...      ...   
2697     cognitive robotics high res      139    video   
2698     cognitive robotics high res      140    video   
2699     cognitive robotics high res      141    video   
2700     cognitive robotics high res      142    video   
2701     cognitive robotics high res      143    video   

                                      file_name        start          end  
0     theory_of_computation_high_res_000000.mp4     0.000000    34.601233  
1     theory_of_computation_high_res_000001.mp4    34.601233    64.230833  
2     theory_of_computation_high_res_000002.mp4    64.230833    93.860433  

# Vector Databse

In [60]:
from pinecone import Pinecone, SchemaBuilder
from dotenv import load_dotenv
import os

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))



### Create Index

In [12]:
schema = (
    SchemaBuilder()
    .add_dense_vector_field(name="embedding", dimension=768, metric="cosine")
    .build()
)

if not pc.indexes.exists("video-vectors"):
    print("Creating index...")
    pc.create_index("video-vectors", schema=schema)

index = pc.Index("video-vectors")

### pass vido file into embedder and insert into database vector

In [ ]:
for video in video_metadata:
    file_path = queue_dir / video["file_name"]
    video_id = video["video_id"]
    clip_id = video["clip_id"]
    id = f"{video_id}_{clip_id}"
    if not file_path.exists():
        continue
    video_embedding = model.encode(
        str(file_path),
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
        show_progress_bar=True,
        processing_kwargs={
            "fps": 1,
            "max_frames": 32,
            "add_timestamps": True,
            "max_soft_tokens": 280,
            "overflow_strategy": "uniform",
        }
    ).reshape(-1)
    subprocess.run([
        "rm", "-rf", str(file_path)
    ])
    index.documents.upsert(
        namespace="videos",
        documents=[
            {
                "_id": id,
                "embedding": video_embedding.tolist(),
                "video_id": str(video_id),
                "clip_id": int(clip_id),
                "modality": "video",
                "start": float(video["start"]),
                "end": float(video["end"]),
            }
        ],
    )
    
    

# Retrieval

In [30]:
def retrieve(query: str, top_k: int = 5):
    query_vector = model.encode(
        query,
        prompt_name="SearchQuery",
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).reshape(-1)
    # Cosine similarity because both sides are normalized.
    scores = embedding_matrix @ query_vector
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for index in top_indices:
        result = {
            **records[index],
            "score": float(scores[index]),
        }
        results.append(result)
    return results

In [35]:
results = retrieve(
    "the lecturer explains supervised learning",
    top_k=10,
)

for result in results:
    print(
        f"Score: {result['score']:.4f} | "
        f"Time: {result['start']:.2f}–{result['end']:.2f}s | "
        f"Clip: {result['clip']} |"
        f"Video: {result['video_path']}"
    )

Score: 0.8324 | Time: 3422.67–3452.67s | Clip: 114 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000114.mp4
Score: 0.8324 | Time: 3422.67–3452.67s | Clip: 114 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000114.mp4
Score: 0.8310 | Time: 3392.67–3422.67s | Clip: 113 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000113.mp4
Score: 0.8310 | Time: 3392.67–3422.67s | Clip: 113 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000113.mp4
Score: 0.8278 | Time: 3242.67–3272.67s | Clip: 108 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000108.mp4
Score: 0.8278 | Time: 3242.67–3272.67s | Clip: 108 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000108.mp4
Score: 0.8277 | Time: 3152.67–3182.67s | Clip: 105 |Video: 222c72dd-8207-42dd-9218-696bc562cc65_000105.mp4
Score: 0.8277 | Time: 3152.67–3182.67s | Clip: 105 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000105.mp4
Score: 0.8277 | Time: 3122.67–3152.67s | Clip: 104 |Video: 9490e964-f624-47a0-a101-02c3d35b211e_000104.mp4
Score: 0.8277 | Time: 3122.67–3152.67

# Testing & Evaluation